# Day 9 — Investor Objection Simulator & Venture Copilot (Layer 8)

The platform reasons like a data-driven VC: it raises the objections an investor would, scores
the risks, builds a SWOT, reads the category and competition, and recommends actions — every
section a reframing of a **real** signal from Days 4–8. Logic in `src/copilot/`.

**Honesty:** deterministic templates, **no LLM** (can't hallucinate a competitor or a market
number); founder / market-size / named-competitor / financial analyses are **deferred** — no
data.

In [1]:
import sys, os, json, warnings, logging
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir(os.path.abspath(".."))
sys.path.insert(0, os.getcwd())
warnings.filterwarnings("ignore"); logging.disable(logging.WARNING)
import pandas as pd, numpy as np

# One warm stack (Day-4 model + Day-5 SHAP + Day-7 frame + Day-8 counterfactual). No faiss.
from src.copilot.pipeline import VentureCopilotPipeline
cop = VentureCopilotPipeline()

# A high-risk demo campaign (Technology, large goal) -> a rich investor brief.
df = pd.read_csv("data/processed/kickstarter_processed.csv"); ti = np.load("artifacts/splits/test_index.npy")
row = df.iloc[ti][(df.iloc[ti]["main_category"]=="Technology") &
                  (df.iloc[ti]["usd_goal_real"]>25000)].sample(1, random_state=3).iloc[0].to_dict()
brief = cop.analyze(row)                    # the whole brief, ONE call
print("CAMPAIGN: %s | %s | goal $%.0f" % (row["name"], row["main_category"], row["usd_goal_real"]))
print("funding probability %.0f%% -> %s (decision bar %.0f%%)" %
      (brief["funding_probability"]*100, brief["predicted_label"], brief["decision_threshold"]*100))

CAMPAIGN: MUV - Clean, safe filtered water adapted to your lifestyle | Technology | goal $30000
funding probability 22% -> failed (decision bar 33%)


## Phase A — Understanding VC Thinking

Day 9 **reframes**, it doesn't compute new: prediction → the odds, SHAP → objections/risks,
patterns → evidence, counterfactuals → the fix. So an objection, a risk, a SWOT weakness, and a
lever are the same signal four ways. Notes: `docs/day9_phaseA_vc_thinking_notes.md`.

## Phase B — Investor Objections

Concerns from structured dimensions (goal / category / timing / cohort / odds), each with
severity + evidence + a suggested response from a Day-8 counterfactual.

In [2]:
for o in brief["investor_objections"][:4]:
    print("[%s] %s" % (o["severity"], o["concern"]))
    print("    -> %s\n" % o["suggested_response"])

[High] Your $30,000 goal is high for Technology, where funded campaigns typically ask a median of $10,000. Why this ask — and can you hit it?
    -> Lower your funding goal from $30,000 to $15,000 — the model estimates about +1 points (23%).

[High] Among campaigns most like yours (Technology, a large goal), only 22% funded. What sets you apart from the 7,940 that came before?
    -> Lower your funding goal from $30,000 to $15,000 — the model estimates about +1 points (23%).

[High] Your 29-day campaign sits in a band that funds at 22% (lift 0.54). Would a shorter campaign convert better?
    -> Add a number to your title (e.g. an edition or volume number) — the model estimates about +3 points (25%).

[High] Technology campaigns historically fund at 24% (lift 0.59) — below the 40% baseline. How do you overcome the category headwind?
    -> Add a number to your title (e.g. an edition or volume number) — the model estimates about +3 points (25%).



## Phase C — Risk Analysis

The same signals as a quantified risk register; fabricated risk categories deferred.

In [3]:
r = brief["risk_analysis"]
print("failure risk %.0f%% | overall: %s\n" % (r["failure_risk"]*100, r["overall_risk_level"]))
for risk in r["risks"][:4]:
    print("  [%s] %-22s %s" % (risk["severity"], risk["category"], risk["historical_evidence"]))
print("\n  deferred:", list(r["deferred_risks"].keys()))

failure risk 78% | overall: High

  [High] financial_goal_risk    Funded Technology campaigns asked a median $10,000.
  [High] funding_risk           Calibrated model: 22% funding probability.
  [High] cohort_risk            Cohort funding rate 22% vs 40% (n=7,940).
  [High] timing_risk            Duration '16-30d': 22% vs 40% (n=13,077).

  deferred: ['technology_risk', 'founder_team_risk', 'unit_economics_risk', 'regulatory_risk', 'market_size_risk']


## Phase D — SWOT

Strengths/Weaknesses from SHAP, Opportunities from counterfactual levers, Threats from
headwinds; market/competitor threats deferred.

In [4]:
sw = brief["swot"]
for quad in ["strengths","weaknesses","opportunities","threats"]:
    pts = [i["point"] for i in sw[quad][:3]]
    print("%-14s %s" % (quad.upper()+":", " | ".join(pts)))

STRENGTHS:     A high title word count | A long title length | The launch day of the week
WEAKNESSES:    A light use of capital letters in the title | A large funding goal | A typical project category
OPPORTUNITIES: Add a number to your title (e.g. an edition or volume number) | Lower your funding goal from $30,000 to $15,000
THREATS:       Category headwind: Technology funds below baseline | Weak cohort: similar campaigns mostly fail | Timing headwind: this campaign length underperforms


## Phase E — Category & Segment Analysis

The honest "market" read: own-data category funding rate, trend (saturation), crowding — no
TAM/CAGR.

In [5]:
c = brief["category_analysis"]
print("%s: %.0f%% funded (lift %.2f) | %s | opportunity %s | timing %s" %
      (c["category"], c["category_funding_rate"]*100, c["category_lift"], c["crowding"],
       c["market_opportunity"], c["market_timing"]))
print(c["summary"])

Technology: 24% funded (lift 0.59) | Crowded | opportunity Challenging | timing Unfavorable
The Technology segment has a 24% historical funding rate (lift 0.59, challenging opportunity), across 21,646 campaigns (crowded). Funding rates have declined over time, making market timing unfavorable. (Within-Kickstarter read; external market size/CAGR not available.)


## Phase F — Category Competition

Competition = other campaigns in the category. What winners do differently, and where you sit.
Named competitors/moat deferred.

In [6]:
cp = brief["competition_analysis"]
print("intensity: %s (%d campaigns)" % (cp["competitive_intensity"]["level"], cp["competitive_intensity"]["n_campaigns"]))
print("winners:", cp["what_winners_do"][0] if cp["what_winners_do"] else "-")
print("your position:", cp["your_position"]["positioned_like"], "->", cp["your_position"]["notes"][0] if cp["your_position"]["notes"] else "")

intensity: Crowded (21646 campaigns)
winners: Ask for less: funded campaigns' median goal is $10,000 vs $20,000 for those that failed.
your position: strugglers -> Your $30,000 goal is 3.0x the winners' median ($10,000) — priced like the campaigns that failed.


## Phase G — Strategic Recommendations

Tactical actions (Day-8 levers) + honest strategic guidance (rescope/delay vs prioritize,
from optimized probability vs the threshold).

In [7]:
rc = brief["recommendations"]
print("current %.0f%% -> optimized %.0f%% (bar %.0f%%)\n" %
      (rc["current_probability"]*100, rc["optimized_probability"]*100, rc["threshold"]*100))
for rec in rc["recommendations"]:
    imp = " [%s]" % rec["estimated_impact"] if rec["estimated_impact"] else ""
    print("  P%d (%s) %s%s" % (rec["priority"], rec["type"], rec["recommendation"][:88], imp))
print("\nGUIDANCE:", rc["overall_guidance"])

current 22% -> optimized 27% (bar 33%)

  P1 (action) Add a number to your title (e.g. an edition or volume number) [+3 points]
  P2 (action) Lower your funding goal from $30,000 to $15,000 [+2 points]
  P3 (strategic) Even with the changes above, the model estimates only ~27% — below the 33% funding bar. 
  P4 (strategic) Timing headwind: funding rates in Technology have declined over time — factor this into 

GUIDANCE: Prioritize: add a number to your title (e.g. an edition or volume number). Applying all recommended changes, the plan reaches ~27%, still below the 33% bar (from 22%).


## Phase H — Venture Copilot Pipeline (coherence)

One `analyze()` produced every section above from one shared signal set — so they agree. The
high goal is the top objection, the top risk, a SWOT weakness, the competitive positioning,
*and* the top fix.

In [8]:
print("brief sections:", list(brief.keys()))
print("\ncoherence check — 'goal' across sections:")
print("  objection:", brief["investor_objections"][0]["dimension"])
print("  risk     :", brief["risk_analysis"]["risks"][0]["category"])
print("  position :", brief["competition_analysis"]["your_position"]["positioned_like"])

brief sections: ['campaign', 'funding_probability', 'predicted_label', 'decision_threshold', 'investor_objections', 'risk_analysis', 'swot', 'category_analysis', 'competition_analysis', 'recommendations', 'honesty_note']

coherence check — 'goal' across sections:
  objection: goal_size
  risk     : financial_goal_risk
  position : strugglers


## Phase I — Serialization

The copilot is a config over Days 4–8; persisted as JSON with a content hash + dependency
hashes (records `no_llm=True`), `verify()` detects drift.

In [9]:
from src.copilot.serialization import CopilotSerializer
ser = CopilotSerializer().build(); ser.save()
print(ser.describe())
print("verify:", ser.verify()["all_ok"])

Venture Copilot v1.0.0 (content 317022c0d7a5…, deterministic=True, no_llm=True): 5 objection dims, 5 risk categories; 4 deps hashed, 10 dimensions deferred.
verify: True


## Day 9 complete — Layer 8 ✅

Investor objections → risk register → real-signal SWOT → category & competition analysis →
strategic recommendations → one coherent copilot brief → content-hashed serialization. Every
section reframes a real Day-4–8 signal via deterministic templates (no LLM); founder, market,
named-competitor, and financial analyses are deferred. The VC-copilot demo, done truthfully.